# soulclip on Kaggle — free AI video from a scene script

Runs **LTX-Video** on Kaggle's free **2x T4** GPUs and stitches the clips
into a finished film. No API key, no payment, no GPU needed on your side.

## Before you run anything

In the **right sidebar**:
1. **Session options > Accelerator > GPU T4 x2**
2. **Internet > On**  (needed to download the model)

Then Run All, or run the cells one at a time.

## What to expect

Cell 8 is set to **6 clips (~30 seconds of film)** so you get a real result
in a few minutes. It prints a measured per-clip time and an ETA — use that
to decide whether to go to 60 clips for a full 5-minute film.


## 1. Confirm the GPUs

If this fails, the accelerator is not set. Fix it in the sidebar and rerun.


In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

import torch
GPUS = torch.cuda.device_count()
assert GPUS > 0, 'No GPU! Sidebar > Session options > Accelerator > GPU T4 x2'
print(f'\n{GPUS} GPU(s) available')
if GPUS < 2:
    print('Only 1 GPU — it will still work, just about twice as slow.')


## 2. Install and fetch the bot

Takes 2-3 minutes. Needs Internet switched on.


In [ ]:
!pip install -q -U diffusers transformers accelerate imageio-ffmpeg

import os, subprocess
REPO = '/kaggle/working/soul_exter'
BRANCH = 'arena/019f98a2-soul-exter'

if not os.path.isdir(REPO):
    subprocess.run(['git','clone','-q','--branch',BRANCH,
                    'https://github.com/Naserkhan07/soul_exter.git', REPO],
                   check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)

os.chdir(REPO)
print('bot ready at', REPO)

# Fail loudly now rather than three cells later.
import importlib.util
assert importlib.util.find_spec('diffusers'), 'diffusers did not install'
print('diffusers installed')


## 3. Your script

Label scenes `Scene 1:`, `Scene 2:` and so on.

**Prompt tips:** describe the *camera* and the *motion*, not just the
subject — "slow dolly in", "waves crash", "hair moves in the wind". Repeat
character details in every scene; the model remembers nothing between clips.


In [ ]:
script = '''
Scene 1: A lighthouse on a black rock headland at dusk, its beam sweeping
across heavy grey water. Rain streaks sideways. Slow dolly in.

Scene 2: Inside the lantern room, brass fittings glowing warm. An old keeper
in a wool coat winds a mechanism by hand. Firelight flickers.

Scene 3: Waves crash white over a dark reef, spray flung high into the storm.
Handheld camera, violent motion.

Scene 4: A small fishing boat pinned against the rocks, mast broken, a single
lantern swinging wildly on the deck.

Scene 5: The keeper hauls a heavy lever with both hands, straining. The great
beam swings across and holds steady.

Scene 6: Dawn over a calm flat sea, pale gold light. Two figures wrapped in
blankets sit on stone steps, steam rising from tin mugs.
'''

SCRIPT_PATH = '/kaggle/working/script.txt'
with open(SCRIPT_PATH, 'w') as f:
    f.write(script)

!python -m soulclip.cli scenes $SCRIPT_PATH --clip-seconds 5


## 4. Settings

Leave `CLIPS = 6` for the first run. Once you see the measured speed, come
back and set it to 60 for a full ~5 minute film.


In [ ]:
CLIPS  = 6                  # 6 = ~30s film. 60 = ~5 minutes.
WIDTH, HEIGHT = 512, 320    # 768x512 looks better but costs ~2x the time
STYLE  = 'cinematic anime, detailed background art, dramatic lighting'

# Computed in Python — shell $(( )) arithmetic does NOT work inside !commands
TARGET = CLIPS * 5

WORKDIR = '/kaggle/working/film/work'
OUTPUT  = '/kaggle/working/film/film.mp4'

import os
os.makedirs(os.path.dirname(OUTPUT), exist_ok=True)
print(f'{CLIPS} clips x 5.04s = about {CLIPS*5.04:.0f}s of film')
print(f'clips -> {WORKDIR}')
print(f'film  -> {OUTPUT}')


## 5. Generate

With 2 GPUs the scenes are split in half and run in parallel.

**The second worker waits 4 minutes before starting.** Kaggle has only
~13 GB of system RAM and `from_pretrained` holds the whole checkpoint
there before moving it to the GPU — two workers loading at the same
moment blows past it and Kaggle restarts the notebook. Staggering means
only one is in that window at a time.

If you still get *"tried to allocate more memory than is available"*,
set `PARALLEL = False` in the cell below. That halves throughput but
uses one model instead of two.

The first run downloads the model (a few GB, once per session), so
expect several quiet minutes before anything appears. After each clip:

```
      avg 43s/clip · 4 left · ~3 min to go
```

That average is **your real measured speed**.

If the session dies, just run this cell again — finished clips are reused.


In [ ]:
import subprocess, os, math, threading, time

# Kaggle gives ~13 GB of SYSTEM RAM, shared by both workers.
# from_pretrained holds the whole checkpoint in CPU RAM before
# moving it to the GPU, so two workers loading at the same moment
# peak at twice that and the kernel gets OOM-killed. Staggering
# the start means only one is in that window at a time.
STAGGER_SECONDS = 240   # set 0 if you have plenty of RAM
PARALLEL = GPUS > 1     # set False to force a single worker

if PARALLEL and CLIPS > 1:
    half = math.ceil(CLIPS / 2)
    ranges = [(1, half), (half + 1, CLIPS)]
else:
    ranges = [(1, CLIPS)]
ranges = [(lo, hi) for lo, hi in ranges if lo <= hi]

results = {}

def worker(gpu, lo, hi, delay):
    if delay:
        print(f'[gpu{gpu}] waiting {delay}s so the first worker '
              f'finishes loading the model', flush=True)
        time.sleep(delay)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    cmd = ['python','-m','soulclip.cli','render', SCRIPT_PATH,
           '--provider','ltx','--clip-seconds','5',
           '--max-scenes', str(CLIPS), '--target', str(TARGET),
           '--width', str(WIDTH), '--height', str(HEIGHT),
           '--style', STYLE, '--workdir', WORKDIR, '-o', OUTPUT]
    if len(ranges) > 1:
        cmd += ['--scenes', f'{lo}-{hi}']
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        # Progress bars are noisy across two workers; keep the
        # lines that actually say something.
        if 'it/s' in line and '100%' not in line:
            continue
        print(f'[gpu{gpu}] {line}', end='', flush=True)
    p.wait()
    results[gpu] = p.returncode

start = time.time()
threads = [threading.Thread(target=worker,
                            args=(g, lo, hi, g * STAGGER_SECONDS))
           for g, (lo, hi) in enumerate(ranges)]
for t in threads: t.start()
for t in threads: t.join()

print(f'\ngeneration finished in {(time.time()-start)/60:.1f} min')
for gpu, code in sorted(results.items()):
    print(f'  gpu{gpu}: exit {code}' + ('' if code == 0 else '  <-- failed'))


## 6. Stitch into the final film

Run without `--scenes` so every clip is picked up and merged. This reuses
all the clips — it does not regenerate anything.


In [ ]:
!python -m soulclip.cli render $SCRIPT_PATH \
    --provider ltx --clip-seconds 5 \
    --max-scenes $CLIPS --target $TARGET \
    --width $WIDTH --height $HEIGHT \
    --workdir $WORKDIR -o $OUTPUT

import os
print()
if os.path.exists(OUTPUT):
    print(f'DONE: {OUTPUT}  ({os.path.getsize(OUTPUT)/1e6:.1f} MB)')
else:
    print('No output file — check the errors above.')


## 7. Watch it


In [ ]:
from IPython.display import HTML
from base64 import b64encode

with open(OUTPUT, 'rb') as f:
    data = b64encode(f.read()).decode()
HTML(f'<video width=640 controls src="data:video/mp4;base64,{data}"></video>')


## 8. Save it before you leave

`/kaggle/working` is **wiped when the session ends** unless you save. Either
click **Save Version** (top right), or download the file from the Output tab
in the right sidebar.


In [ ]:
import shutil
shutil.copy(OUTPUT, '/kaggle/working/my_film.mp4')
print('Saved to /kaggle/working/my_film.mp4')
print('Download it from the Output panel in the right sidebar,')
print('or click Save Version to keep it with the notebook.')


---
## Going to a full 5 minutes

Set `CLIPS = 60` in cell 8 and rerun cells 5 and 6.

| Clips | Film length |
|---|---|
| 6 | ~30 s |
| 60 | ~5m02s |

**How long it takes:** read the `avg Ns/clip` line from your own run and
multiply by 60. With 2 GPUs the wall-clock is roughly half that. I cannot
give you a reliable figure in advance — published numbers for T4-class
hardware vary by several times, which is exactly why the bot measures
itself.

Rough bracket at 512x320 on 2x T4: **25-45 minutes** for 60 clips.

### Kaggle free tier

| | |
|---|---|
| Weekly GPU | 30 hours, guaranteed |
| Session | 9-12 hours |
| Cost | Rs 0 |

That is roughly 30-40 five-minute films a week for nothing.

> **Account warning:** Kaggle has banned accounts that only ever consume GPU
> hours without taking part in the community. Use it in moderation.

### Troubleshooting

| Problem | Fix |
|---|---|
| `No GPU!` | Sidebar > Session options > Accelerator > GPU T4 x2 |
| Clone or pip fails | Sidebar > Internet > On |
| Out of memory | Lower to `WIDTH, HEIGHT = 384, 256` |
| Session died mid-run | Just rerun cell 5 — finished clips are reused |
| Quality too soft | Raise to `768, 512` (about 2x slower) |

### Honest expectations

LTX-Video 2B distilled is the fastest usable open model, not the best
looking. Output is low-resolution and well short of paid Kling or Veo.
Characters will not stay consistent between shots — that is a limitation of
the model, not this pipeline. Repeating character details in every scene
helps a little.
